In [1]:
import pandas as pd

In [2]:
#import data from matching
fuzzy_raw = pd.read_csv('data/fuzzy_test_class_only.csv', index_col=[0])
fuzzy_raw.tail()

,Dyad,Session,Time_begin,P1,P2,SD,QE,SV,PR,HD,Match
1649,10,2,00:46:24.900000,nice snails,NaN,NaN,NaN,x,x,NaN,"[('nice snails', 100), [10, 2, '00:46:24.938']]"
1650,10,2,00:46:26.600000,NaN,thanks I only did one hand,NaN,NaN,x,NaN,NaN,"[('thanks I only did one hand', 100), [10, 2, ..."
1651,10,2,00:46:28.400000,nice,NaN,NaN,NaN,x,x,NaN,"[('nice', 100), [10, 2, '00:46:28.378']]"
1652,10,2,00:46:29.200000,NaN,I was too lazy okay,NaN,NaN,x,NaN,NaN,"[('I was too lazy okay', 100), [10, 2, '00:46:..."
1653,10,2,00:46:30.900000,yeah that's me,NaN,NaN,NaN,x,NaN,NaN,"[(""yeah that's me"", 100), [10, 2, '00:46:30.84..."


In [3]:
#Breaking the 'Match' column into columns ['match', 'score', 'dyad', 'session', 'time_begin']
#putting the result in fuzzy_processed

def fuzzy_column_splitting(x):
    """
    To be used in an apply().
    As you can see above the Match column in a list of [ (match,score) , [Dyad, Session, Time_begin] ].
    So it just retrieve all these elements and put them into 5 columns.
    """
    x=eval(x)
    res = pd.Series([
        x[0][0], x[0][1] , x[1][0], x[1][1], x[1][2]
        ])
    return res
#    try : ##I use to have issues, I leave this here just in case
#        x=eval(x)
#        res = pd.Series([
#            x[0][0], x[0][1] , x[1][0], x[1][1], x[1][2]
#            ])
#        return res
#    except : return x

fuzzy_processed = fuzzy_raw.copy()
print(fuzzy_raw.shape)
print(fuzzy_processed.shape)

#retrieving all information from fuzzy column ('Match') and adding them in de dataframe
fuzzy_processed[['match', 'score', 'dyad', 'session', 'time_begin']] = fuzzy_raw.Match.apply(fuzzy_column_splitting).values
fuzzy_processed.drop(columns=['Match'], inplace=True)
print(fuzzy_raw.shape)
print(fuzzy_processed.shape)
fuzzy_processed.head()

(2170, 11)
(2170, 11)
(2170, 11)
(2170, 15)


,Dyad,Session,Time_begin,P1,P2,SD,QE,SV,PR,HD,match,score,dyad,session,time_begin
9322,3,1,00:00:43.600000,this is weird I think,NaN,x,NaN,NaN,NaN,NaN,this is weird I,95,3,1,00:00:45.135
9324,3,1,00:00:48.600000,NaN,maybe if you just hit undo button or something,NaN,NaN,NaN,NaN,x,maybe if you just hit undo button or something,100,3,1,00:00:51.000
9347,3,1,00:02:17.000000,NaN,I just can't see the problem,NaN,NaN,NaN,NaN,x,I see green on most of the pages I just can't ...,90,3,1,00:02:15.080
9349,3,1,00:02:30.600000,I don't know if you saw that,NaN,x,NaN,NaN,NaN,NaN,I don't know if you saw that,100,3,1,00:02:30.730
9364,3,1,00:03:16.300000,okay so pause filler so what's your name,NaN,NaN,x,NaN,NaN,NaN,okay so pause filler so what's your name,95,3,1,00:03:16.336


In [4]:
#df_temp to be merged with the full dataset
df_temp = pd.DataFrame(fuzzy_processed[['score', 'SD', 'QE', 'SV', 'PR', 'HD']].values, columns=['score', 'SD', 'QE', 'SV', 'PR', 'HD'])

#creating an 'id' column and putting it to index
id = fuzzy_processed.match.astype(str) \
    + ' - ' + fuzzy_processed.dyad.astype(str) \
    + ' - ' + fuzzy_processed.session.astype(str) \
    + ' - ' + pd.to_datetime(
        fuzzy_processed.time_begin,
        format='%H:%M:%S.%f'
    ).dt.strftime("%H:%M:%S.%f").str.split('.', expand=True)[0]
df_temp['id'] = id.reset_index(drop=True)
df_temp.set_index('id', drop=True, inplace=True)

df_temp.head()

,score,SD,QE,SV,PR,HD
id,,,,,,
this is weird I - 3 - 1 - 00:00:45,95,x,NaN,NaN,NaN,NaN
maybe if you just hit undo button or something - 3 - 1 - 00:00:51,100,NaN,NaN,NaN,NaN,x
I see green on most of the pages I just can't see the problem - 3 - 1 - 00:02:15,90,NaN,NaN,NaN,NaN,x
I don't know if you saw that - 3 - 1 - 00:02:30,100,x,NaN,NaN,NaN,NaN
okay so pause filler so what's your name - 3 - 1 - 00:03:16,95,NaN,x,NaN,NaN,NaN


In [5]:
#importing full dataset into 'full'
full = pd.read_csv("data/aligned_transcript_nolabel.csv", index_col=[0])

#reproducing the id column (I chose to make it with P1 and P2 and then merge it)

#id1 with P1
full['id1'] = full.P1.astype(str) \
    + ' - ' + full.Dyad.astype(str) \
    + ' - ' + full.Session.astype(str) \
    + ' - ' + pd.to_datetime(full.Time_begin_ts, format='%Y-%m-%d %H:%M:%S.%f').dt.strftime("%H:%M:%S.%f").str.split('.', expand=True)[0]
#id2 with P2
full['id2'] = full.P2.astype(str) \
    + ' - ' + full.Dyad.astype(str) \
    + ' - ' + full.Session.astype(str) \
    + ' - ' + pd.to_datetime(full.Time_begin_ts, format='%Y-%m-%d %H:%M:%S.%f').dt.strftime("%H:%M:%S.%f").str.split('.', expand=True)[0]

#merging id1 and id2 into id
full['id'] = full[['id1', 'id2']].apply(
    lambda x : x.id1 if x.id1[:3] != 'nan' else x.id2, #empty utt always starts with 'nan'
    axis=1
    )
full.drop(columns=['id1', 'id2'], inplace=True)

full.head()

,Dyad,Session,Begin Time - hh:mm:ss.ms,End Time - hh:mm:ss.ms,Duration - hh:mm:ss.ms,P1,P2,Time_begin_ts,id
0,3,1,00:00:00.250,00:00:01.320,00:00:01.070,so you can hear me now,NaN,1900-01-01 00:00:00.250,so you can hear me now - 3 - 1 - 00:00:00
1,3,1,00:00:01.570,00:00:02.390,00:00:00.820,NaN,yeah I can hear you now,1900-01-01 00:00:01.570,yeah I can hear you now - 3 - 1 - 00:00:01
2,3,1,00:00:04.650,00:00:05.230,00:00:00.580,that's good,NaN,1900-01-01 00:00:04.650,that's good - 3 - 1 - 00:00:04
3,3,1,00:00:08.100,00:00:08.840,00:00:00.740,oh loading,NaN,1900-01-01 00:00:08.100,oh loading - 3 - 1 - 00:00:08
4,3,1,00:00:09.500,00:00:10.355,00:00:00.855,did I hit something,NaN,1900-01-01 00:00:09.500,did I hit something - 3 - 1 - 00:00:09


In [6]:
#safety checks
print(len(df_temp))
print(len(set(df_temp.index)))
print(len(set(full.id).intersection(set(df_temp.index))))
#at least the id column works but some matches have possibly been assigned to multiple rows or the inverse
#(this has to be dealt with by hand)

2170
2083
2083


In [7]:
#make result
result = full.merge(
    df_temp[['score','SD','QE','SV','PR','HD']],
    how='left',
    on='id'
    )
result.drop(columns=['id'], inplace=True)

#############
## Save it ##
#############

#result.to_csv('result.csv')

result.head()
# to see non empty cat rows run
#result.loc[~result.score.isna()]

,Dyad,Session,Begin Time - hh:mm:ss.ms,End Time - hh:mm:ss.ms,Duration - hh:mm:ss.ms,P1,P2,Time_begin_ts,score,SD,QE,SV,PR,HD
0,3,1,00:00:00.250,00:00:01.320,00:00:01.070,so you can hear me now,NaN,1900-01-01 00:00:00.250,NaN,NaN,NaN,NaN,NaN,NaN
1,3,1,00:00:01.570,00:00:02.390,00:00:00.820,NaN,yeah I can hear you now,1900-01-01 00:00:01.570,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,00:00:04.650,00:00:05.230,00:00:00.580,that's good,NaN,1900-01-01 00:00:04.650,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1,00:00:08.100,00:00:08.840,00:00:00.740,oh loading,NaN,1900-01-01 00:00:08.100,NaN,NaN,NaN,NaN,NaN,NaN
4,3,1,00:00:09.500,00:00:10.355,00:00:00.855,did I hit something,NaN,1900-01-01 00:00:09.500,NaN,NaN,NaN,NaN,NaN,NaN
